# Part 4f — Interdiction and defence

### Defender–attacker–defender: the rigorous version of "stage-targeted embargo"

Part 4e treated policy as a *parameter* — set a tariff, watch firms respond. That is the right model
for an instrument applied openly and held fixed. It is the wrong model for a deliberate disruption,
where the adversary chooses **where** to strike precisely because it hurts, and the defender chooses
**what to protect** anticipating that choice.

That is a three-level problem:

$$\max_{\text{defend}} \;\; \min_{\text{attack}} \;\; \max_{\text{operate}} \;\; \text{delivered volume}$$

Read outward: the operator routes around whatever damage exists; the attacker picks the damage that
survives the best routing; the defender fortifies against the attacker's best move. Each level
anticipates everything inside it.

### Why this is not just a hard MILP

You cannot solve a trilevel program by writing it down and calling `optimize()`. The standard route
collapses the inner two levels using duality, and handles the outer level by decomposition. This
notebook does the collapse **exactly** and the outer level by **enumeration**, which is honest at
this size and makes the structure visible.

### The duality that makes it work

The operator solves a max-flow. By max-flow/min-cut, its optimal value equals the minimum capacity
of an $s$–$t$ cut. So

$$\min_{\text{attack}} \; \underbrace{\max_{\text{operate}} \text{flow}}_{\text{an LP}}
\;=\; \min_{\text{attack}} \; \min_{\text{cut}} \; \text{cut capacity}
\;=\; \min_{\text{attack},\,\text{cut}} \;(\cdot)$$

Two nested minimisations are **one** minimisation. The attacker's problem becomes a single MILP —
no bilevel machinery, no big-M on a KKT block, no complementarity. This is the cleanest instance in
the whole series of a structural property buying exactness.

## Formulation reference

### Operator (inner) — max flow

$$\max \sum_{a \in \delta^+(s)} f_a \quad\text{s.t.}\quad
0 \le f_a \le \kappa_a(1-\zeta_a), \quad \text{conservation at every intermediate node}$$

$\zeta_a = 1$ if arc $a$ is interdicted.

### Attacker (middle) — min cut with interdiction

Dualising the operator gives node potentials $\pi_i$ and cut indicators $\gamma_a$:

$$\min \;\; \sum_a \kappa_a\, \omega_a$$
$$\gamma_{ij} - \pi_i + \pi_j \ge 0 \;\;\forall (i,j), \qquad \pi_s - \pi_t \ge 1$$
$$\omega_a \;\ge\; \gamma_a - \zeta_a, \qquad \omega_a \ge 0$$
$$\sum_a \zeta_a \le B_{\text{atk}}, \qquad \zeta_a \le 1 - \phi_a$$

The $\omega$ substitution is the linearisation. The product $\kappa_a(1-\zeta_a)\gamma_a$ would be
bilinear; instead $\omega_a \ge \gamma_a - \zeta_a$ with $\omega \ge 0$ and a minimised nonnegative
cost forces $\omega_a = \gamma_a$ when the arc survives and lets $\omega_a = 0$ when it is cut.
**No big-M appears anywhere**, which is why this formulation is numerically well behaved.

$\phi_a = 1$ marks a fortified arc, which the attacker may not touch.

### Defender (outer)

$$\max_{\phi:\;\sum_a \phi_a \le B_{\text{def}}} \;\; \big[\text{attacker's optimal value under } \phi\big]$$

Enumerated here. At production scale this is where LLNL's Best Response Intersection decomposition
belongs — an outer defence problem generating candidate fortifications against an inner
attacker's-best-response oracle.

## 1. Setup and network

In [ ]:
import os, itertools
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt

for cand in ("gurobi.lic", os.path.join("..", "gurobi.lic")):
    if os.path.exists(cand):
        os.environ["GRB_LICENSE_FILE"] = os.path.abspath(cand); break
plt.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

# A four-stage chain across three regions, plus a super-source and super-sink.
REGS   = ["CHN", "ALY", "ROW"]
STAGES = ["MINE", "REF", "CAM", "CELL"]

def n(stage, reg): return f"{stage}:{reg}"

NODESET = ["SRC"] + [n(s, r) for s in STAGES for r in REGS] + ["SNK"]

CAP = {}
# reserves into mining - ROW and CHN are endowed, ALY less so
for r, k in [("CHN", 55), ("ALY", 18), ("ROW", 70)]:
    CAP[("SRC", n("MINE", r))] = k
# stage-to-stage, all region pairs; intra-region generous, cross-region thinner
for i in range(len(STAGES)-1):
    s1, s2 = STAGES[i], STAGES[i+1]
    for r1 in REGS:
        for r2 in REGS:
            CAP[(n(s1, r1), n(s2, r2))] = 40 if r1 == r2 else 16
# delivery to demand
for r, k in [("CHN", 45), ("ALY", 40), ("ROW", 35)]:
    CAP[(n("CELL", r), "SNK")] = k

ARCS = list(CAP.keys())

# The super-source and super-sink are modelling artifacts - a reserve base and a demand
# aggregate, not physical links anyone can sever. Only stage-to-stage arcs are real
# supply-chain connections, so only those are interdictable. Without this restriction the
# attacker simply cuts the three source arcs and the problem is trivial.
ATTACKABLE = [a for a in ARCS if a[0] != "SRC" and a[1] != "SNK"]
print(f"{len(NODESET)} nodes, {len(ARCS)} arcs, {len(ATTACKABLE)} interdictable")


## 2. The operator's problem, solved directly

The baseline: no attack, no defence. This is the volume the chain can deliver when nothing is wrong.

In [ ]:
def max_flow(interdicted=frozenset()):
    m = gp.Model(); m.Params.OutputFlag = 0
    f = {a: m.addVar(lb=0.0, ub=(0.0 if a in interdicted else CAP[a])) for a in ARCS}
    for v in NODESET:
        if v in ("SRC", "SNK"):
            continue
        m.addConstr(gp.quicksum(f[a] for a in ARCS if a[1] == v)
                    == gp.quicksum(f[a] for a in ARCS if a[0] == v), name=f"bal_{v}")
    m.setObjective(gp.quicksum(f[a] for a in ARCS if a[0] == "SRC"), GRB.MAXIMIZE)
    m.optimize()
    return m.ObjVal, {a: f[a].X for a in ARCS}

base, _ = max_flow()
print("undisrupted throughput: %.1f" % base)


## 3. The attacker's best response — exact, as a single MILP

`fortified` is the defender's choice. The returned value is what the operator can still deliver
after the worst admissible attack.

In [ ]:
def attacker_best_response(budget, fortified=frozenset()):
    m = gp.Model(); m.Params.OutputFlag = 0
    pi  = m.addVars(NODESET, lb=0.0, name="pi")
    gam = m.addVars(ARCS, lb=0.0, name="gamma")
    om  = m.addVars(ARCS, lb=0.0, name="omega")
    zet = m.addVars(ARCS, vtype=GRB.BINARY, name="zeta")

    m.addConstrs((gam[a] - pi[a[0]] + pi[a[1]] >= 0 for a in ARCS), name="cut")
    m.addConstr(pi["SRC"] - pi["SNK"] >= 1, name="sep")
    m.addConstrs((om[a] >= gam[a] - zet[a] for a in ARCS), name="lin")
    m.addConstr(gp.quicksum(zet[a] for a in ATTACKABLE) <= budget, name="budget")
    for a in ARCS:
        if a not in ATTACKABLE or a in fortified:
            m.addConstr(zet[a] == 0)
    m.setObjective(gp.quicksum(CAP[a]*om[a] for a in ARCS), GRB.MINIMIZE)
    m.optimize()
    attack = frozenset(a for a in ARCS if zet[a].X > 0.5)
    return m.ObjVal, attack


### Verify the collapse

The MILP claims to equal `max_flow` under the attack it selects. If duality has been applied
correctly those two numbers agree exactly. A sign error here still produces a plausible-looking
attack — this check is the only thing that catches it.

In [ ]:
print(" B | MILP value | direct max-flow | attack")
ok = True
for B in range(0, 5):
    val, atk = attacker_best_response(B)
    direct, _ = max_flow(atk)
    match = abs(val - direct) < 1e-6
    ok &= match
    names = ", ".join(f"{a[0]}->{a[1]}" for a in sorted(atk)) or "(none)"
    print(f" {B} | {val:10.2f} | {direct:15.2f} | {names}")
    assert match, f"duality collapse failed at B={B}: {val} vs {direct}"
print("\nPASS - the min-cut MILP reproduces the operator's max-flow exactly")

## 4. How much damage, per unit of attacker budget

In [ ]:
rows = []
for B in range(0, 7):
    val, atk = attacker_best_response(B)
    rows.append(dict(budget=B, throughput=round(val, 2),
                     loss=round(base - val, 2), loss_pct=round(100*(base-val)/base, 1)))
dmg = pd.DataFrame(rows)
print(dmg.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dmg["budget"], dmg["throughput"], "o-")
ax.axhline(base, ls="--", c="k", lw=1, label="undisrupted")
ax.set_xlabel("arcs the attacker may remove"); ax.set_ylabel("throughput")
ax.set_title("Marginal damage is not constant"); ax.legend()
plt.tight_layout(); plt.show()


## 5. The defender

Enumerate fortification sets of size $B_{\text{def}}$ and take the one that maximises the attacker's
minimum.

**A trap worth recording.** The obvious economy is to consider only arcs the attacker actually
chooses in the undefended problem — fortifying an arc nobody attacks looks like wasted budget. That
reasoning is **wrong**, and §10 catches it: fortification *changes which attacks are rational*, so an
arc that is never attacked today can be exactly the arc worth protecting, because protecting it
forces the attacker onto a more expensive cut. Restricting candidates that way silently converts an
exact method into a heuristic that reports a *worse* optimum than the truth.

The enumeration below therefore ranges over the full interdictable set. `candidates` is retained as
an argument so the restricted version can be run for comparison — but it is not the default.

In [ ]:
def defender(bdef, batk, candidates=None, verbose=True):
    # default: the FULL interdictable set. Restricting to previously-attacked arcs
    # is a heuristic, not an economy - see the note above and Section 10.
    if candidates is None:
        candidates = sorted(ATTACKABLE)
    best = (-1.0, None, None)
    for combo in itertools.combinations(candidates, bdef):
        val, atk = attacker_best_response(batk, fortified=frozenset(combo))
        if val > best[0]:
            best = (val, combo, atk)
    if verbose:
        print(f"candidate arcs considered: {len(candidates)}")
        print(f"fortification sets evaluated: {len(list(itertools.combinations(candidates, bdef)))}")
    return best

BATK = 3
no_def, atk0 = attacker_best_response(BATK)
print("attacker budget %d, no defence -> throughput %.2f (loss %.1f%%)\n"
      % (BATK, no_def, 100*(base-no_def)/base))

for bdef in (1, 2):
    val, fort, atk = defender(bdef, BATK)
    print(f"\ndefend {bdef}: throughput {val:.2f}  (recovers {val-no_def:.2f} of {base-no_def:.2f} lost)")
    print("  fortify:", ", ".join(f"{a[0]}->{a[1]}" for a in fort))
    print("  attacker then hits:", ", ".join(f"{a[0]}->{a[1]}" for a in sorted(atk)))


### Where the second unit of defence goes

The first fortification recovers part of the loss. The second recovers more — but **only if the
search is allowed to consider arcs the attacker is not currently attacking**.

That is not an obvious point, and it is easy to get wrong. Running this same enumeration over the
restricted candidate set (arcs chosen in the undefended problem) returns *no improvement at all* from
the second unit, which looks like a clean diminishing-returns result and is an artifact. §10
quantifies it: the restriction costs 5 units of throughput and hides the fact that the best
two-arc defence protects one upstream arc **and** one arc further down that nobody attacks today.

The mechanism: fortifying the obvious arc pushes the attacker onto a different cut, and the second
fortification should be aimed at *that* cut, not at the original one. Defence is a response to the
attacker's response — which is the whole reason this is a trilevel problem and not a ranking exercise.


## 6. Which arcs are critical?

Frequency across the attacker's optimal responses at every budget — a ranking of where the chain is
actually fragile, as opposed to where it looks fragile.

In [ ]:
from collections import Counter
cnt = Counter()
for B in range(1, 7):
    _, atk = attacker_best_response(B)
    cnt.update(atk)
crit = pd.DataFrame([{"arc": f"{a[0]} -> {a[1]}", "capacity": CAP[a], "times_chosen": c}
                     for a, c in cnt.most_common()])
print(crit.to_string(index=False))

## 7. Reading this against the LLNL interdiction work

The Yao–Yang–Grappone–Glista–Yuan–Gounaris–Watson model is the same structure at research scale, and
the comparison is worth making precisely.

| | This notebook | LLNL |
|---|---|---|
| Levels | defender–attacker–operator | same |
| Inner collapse | max-flow/min-cut duality | LP duality on a richer operator model |
| Outer solve | enumeration over candidate arcs | **Best Response Intersection** decomposition |
| Periods | single | single (multi-period is their stated future work) |
| Recycling | absent | absent (also their future work) |
| Costs | throughput only | throughput; fortification cost is future work |

Their reported findings — worst-case damage concentrated in intra-China flows and DRC mining, and
midstream vulnerability at the anode stage — are the same *kind* of output as §6 here: a ranking of
arcs by how often an optimising adversary selects them.

**Where the two programmes join.** Their model is single-period with no recycling; the Part 5 core is
multi-period with a closed loop. An interdiction layer over Part 5's arcs would ask a question
neither can currently answer: *does recycling capacity built for cost reasons also harden the chain,
and if so by how much?* That is a genuinely new question, and it needs both halves.

### Honest limitations here

- Throughput is the only objective. Real interdiction has costs on both sides, and a defender
  trading fortification budget against capacity investment is a different and harder problem.
- Enumeration is exact but does not scale; past a few dozen candidate arcs it must be replaced.
- The attacker is omniscient and unconstrained by attribution or escalation risk. That makes this a
  worst-case bound, which is the right thing for planning and the wrong thing for prediction.
- Max-flow ignores cost entirely. Extending to min-cost flow keeps the duality but the cut
  interpretation is no longer as clean.


## 8. Best Response Intersection — replacing enumeration

Enumeration is exact and it does not scale. With $|A|$ candidate arcs and a defence budget
$B_{\text{def}}$ it costs $\binom{|A|}{B_{\text{def}}}$ attacker solves; at 27 arcs and a budget of
four that is 17,550 MILPs. This is the decomposition LLNL uses instead.

### The idea

The defender does not need to guard against every conceivable attack — only against attacks that are
a **best response to some fortification**. Those are generated on demand:

$$
\textbf{NDP:}\quad \max_{\phi,\theta}\; \theta
\quad\text{s.t.}\quad \theta \le \text{flow}_j(\phi)\;\; \forall j \in \mathcal{J},
\qquad \textstyle\sum_a \phi_a \le B_{\text{def}}
$$

$$
\textbf{ABR:}\quad v(\phi^*) \;=\; \min_{\text{attack}} \max_{\text{operate}} \text{flow}
\qquad\text{(§3, already exact)}
$$

Solve NDP, take $\phi^*$, find its worst attack, add that attack to $\mathcal{J}$, repeat. $\theta^*$
falls monotonically (an upper bound — the defender is only guarding against a subset) and
$v(\phi^*)$ gives a lower bound. When they meet, $\phi^*$ is optimal.

### The step that makes the master linear

For a **fixed** attack pattern $z^j$, the capacity surviving on arc $a$ is

$$\bar\kappa_a^{\,j}(\phi) \;=\; \kappa_a\big(1 - z^j_a(1-\phi_a)\big)$$

$z^j$ is a constant, so this is **linear in $\phi$**: an unattacked arc keeps $\kappa_a$, an attacked
arc keeps $\kappa_a\phi_a$ — full capacity if fortified, zero if not. Each retained attack therefore
contributes an ordinary max-flow LP block to the master, and the whole NDP is a MILP with binary
$\phi$ and continuous flows. No bilinearity, no big-M, nothing tuned.

The master grows by one flow block per iteration, which is the cost of the method and the reason it
terminates quickly in practice: few attacks are ever best responses.

In [ ]:
def ndp(attacks, bdef, ub_theta):
    """Defender master: fortify to maximise the worst flow over the retained attacks."""
    m = gp.Model(); m.Params.OutputFlag = 0
    phi = m.addVars(ATTACKABLE, vtype=GRB.BINARY, name="phi")
    th  = m.addVar(lb=0.0, ub=ub_theta, name="theta")
    m.addConstr(gp.quicksum(phi[a] for a in ATTACKABLE) <= bdef, name="budget")

    for j, atk in enumerate(attacks):
        f = m.addVars(ARCS, lb=0.0, name=f"f{j}")
        for a in ARCS:
            if a in atk:                       # attacked: survives only if fortified
                m.addConstr(f[a] <= CAP[a]*phi[a], name=f"cap{j}_{a}")
            else:
                m.addConstr(f[a] <= CAP[a], name=f"cap{j}_{a}")
        for v in NODESET:
            if v in ("SRC", "SNK"):
                continue
            m.addConstr(gp.quicksum(f[a] for a in ARCS if a[1] == v)
                        == gp.quicksum(f[a] for a in ARCS if a[0] == v))
        m.addConstr(th <= gp.quicksum(f[a] for a in ARCS if a[0] == "SRC"), name=f"theta{j}")

    m.setObjective(th, GRB.MAXIMIZE)
    m.optimize()
    fort = frozenset(a for a in ATTACKABLE if phi[a].X > 0.5)
    return m.ObjVal, fort

In [ ]:
def bri(bdef, batk, max_iter=40, tol=1e-6, verbose=True):
    # seed with the best response to doing nothing
    v0, a0 = attacker_best_response(batk)
    attacks = [a0]
    LB, best_fort = v0, frozenset()
    hist = []
    for it in range(1, max_iter+1):
        UB, fort = ndp(attacks, bdef, ub_theta=base)          # upper bound
        v, atk   = attacker_best_response(batk, fortified=fort)  # true value of this fort
        if v > LB:
            LB, best_fort = v, fort
        hist.append(dict(iter=it, UB=round(UB, 4), LB=round(LB, 4),
                         gap=round(UB-LB, 6), attacks=len(attacks)))
        if verbose:
            print(f"  it {it:2d}  UB {UB:8.3f}  LB {LB:8.3f}  gap {UB-LB:8.5f}  |J|={len(attacks)}")
        if UB - LB < tol:
            break
        if atk in attacks:            # no new information -> converged
            break
        attacks.append(atk)
    return LB, best_fort, pd.DataFrame(hist)

print("BRI, defend 1 vs attacker budget 3:")
bri_val, bri_fort, bri_hist = bri(1, BATK)
print("\nvalue %.2f  fortify: %s" % (bri_val, ", ".join(f"{a[0]}->{a[1]}" for a in bri_fort)))


## 10. Validation — and the candidate-restriction trap

Full enumeration is exact at this size, so it is the ground truth for BRI. The third column repeats
the enumeration over the *restricted* candidate set — only arcs the attacker chooses when nothing is
fortified — to show what that shortcut costs.

In [ ]:
# the restricted candidate set: arcs an unfortified attacker ever picks
seen = set()
for B in range(1, BATK+3):
    _, atk = attacker_best_response(B)
    seen |= set(atk)
restricted = sorted(seen)
print("full interdictable set: %d arcs | restricted set: %d arcs\n" % (len(ATTACKABLE), len(restricted)))

for bd in (1, 2):
    enum_val, enum_fort, _ = defender(bd, BATK, verbose=False)
    b_val,   b_fort,   _   = bri(bd, BATK, verbose=False)
    r_val,   r_fort,   _   = defender(bd, BATK, candidates=restricted, verbose=False)
    ok = abs(enum_val - b_val) < 1e-6
    print(f"defend {bd}:  full enumeration {enum_val:8.3f} | BRI {b_val:8.3f} "
          f"| restricted {r_val:8.3f} | {'MATCH' if ok else 'MISMATCH'}")
    print(f"           full fortifies {sorted(a[0]+'->'+a[1] for a in enum_fort)}")
    print(f"           BRI  fortifies {sorted(a[0]+'->'+a[1] for a in b_fort)}")
    if r_val < enum_val - 1e-6:
        print(f"           [!] the restricted set misses {enum_val-r_val:.2f} of throughput")
    assert ok, "BRI did not reproduce the enumerated optimum"
print("\nPASS - BRI matches full enumeration (fortification sets may differ where tied)")

### Convergence

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(bri_hist["iter"], bri_hist["UB"], "o-", label="upper bound (NDP)")
ax.plot(bri_hist["iter"], bri_hist["LB"], "s-", label="lower bound (ABR)")
ax.set_xlabel("iteration"); ax.set_ylabel("throughput under worst attack")
ax.set_title("BRI closes from both sides"); ax.legend()
plt.tight_layout(); plt.show()
print(bri_hist.to_string(index=False))


### Where enumeration stops being possible

The point of the decomposition. At a defence budget of 4 over the full interdictable set,
enumeration would need $\binom{27}{4}$ attacker solves; BRI needs one per iteration.

In [ ]:
from math import comb
import time

BD = 4
n_cand = len(ATTACKABLE)
print("enumeration would require %s attacker solves" % f"{comb(n_cand, BD):,}")

t0 = time.time()
v4, f4, h4 = bri(BD, BATK, verbose=False)
t4 = time.time() - t0
print("BRI: %d iterations, %.2fs -> throughput %.2f" % (len(h4), t4, v4))
print("fortify:", ", ".join(f"{a[0]}->{a[1]}" for a in sorted(f4)))
print("\nspeedup vs enumeration: roughly %s x" % f"{comb(n_cand, BD)//max(1,len(h4)):,}")


### Why it terminates so quickly

The retained attack set stays small because **most attacks are never a best response to anything**.
The defender is not searching the space of attacks; it is searching the much smaller space of
attacks that are optimal against some fortification. That is the whole content of the word
*intersection* — you only ever need the attacks that sit at the boundary between defensive postures.

The worst case is still exponential. What makes it practical is that the number of *distinct* best
responses is governed by the number of near-tied cuts, not by $\binom{|A|}{B}$.